In [ ]:
# general
import sys 
import gc
from pathlib import Path
# virtual cell
import h5py
from gears import GEARS, PertData
import scanpy as sc
# data 
import numpy as np
import pandas as pd
import torch
# modelling 

# paths
vcc_data_path = Path('data/VCC_filtered.h5ad')
# for each model we will have separate folder with place for loader and model separately 
gears_data_path = Path('models') 
loader_path = Path('loaders/vcc_processed')


## Data preperation

### Assumptions
We have right amount of perturbations (?) with at least > 40 samples per perturbation 
### Format
We create scanpy object and need to be:
- adata.obs
    - rename column: \
    'target_gene' -> 'condition'
    - change category: \
    'non-targeting' in 'condition' column -> 'ctrl'
    - change category: \
    'gene' in 'condition' column -> 'gene' + '+ctrl' - unperturbed
    - add column: \
    add column with cell type 'H1'
- adata.var
    - rename column: \
    'gene_id' -> 'gene_name'


### Loading data

### 3.10 - lack of features
- we need to change names of some operations like log1p which are not present in current file

In [ ]:
# 3.10 only
# repairing file - additional arguments:
with h5py.File('put_data_path_here', "r+") as f:
    # checking value
    print(list(f["uns/log1p"].keys()))  # powinno pokazać 'base'
    
    # getting read of value
    if "base" in f["uns/log1p"]:
        del f["uns/log1p/base"]

    if "mask_var" in f["uns/pca/params"]:
        print("Usuwam uns/pca/params/mask_var")
        del f["uns/pca/params/mask_var"]

gc.collect()

In [ ]:
# loading part of data
adata_big = sc.read_h5ad(vcc_data_path, backed='r')
adata_big

# for later manipulations
mask_ctrl = adata_big.obs['target_gene'] == 'non-targeting'
mask_rest = ~mask_ctrl

In [ ]:
# IMPORTANT - partial loading - do not if loading whole file

# parameters #TODO
genes_amount =  100
probes_per_gene = 50 

# taking random control cells
n_ctrl = 150
ctrl_idx = np.random.choice(
    np.where(mask_ctrl)[0],
    size=min(n_ctrl, mask_ctrl.sum())
    replace=False
)

# applying mask
all_perts = adata_big.obs.loc[mask_rest, 'target_gene'].unique()
# taking random perturbations
chosen_perts = np.random.choice(all_perts, size=min(genes_amount, len(all_perts)), replace=False)

# taking random probes_per_gene cells from each perturbation
rest_idx_list = []
for pert in chosen_perts:
    idxs = np.where(adata_big.obs['target_gene'] == pert)[0]
    take_n = min(probes_per_gene, len(idxs))  # zabezpieczenie, jeśli jest mniej niż 40 komórek
    rest_idx_list.append(np.random.choice(idxs, size=take_n, replace=False))

# joining indices
final_idx = np.concatenate([ctrl_idx] + rest_idx_list)

In [ ]:
# loading data to memory
adata = adata_big.to_memory()

In [ ]:
## preprocessing - w przypadku gdy nie jest podziałane !!!
# sc.pp.normalize_total(adata)
# sc.pp.log1p(adata)

In [ ]:
# creating right non-targeting -> ctrl
adata.obs.rename(columns={'target_gene': 'condition',}, inplace=True)
# converting data from categorical to string
adata.obs['condition'] = adata.obs['condition'].astype(str)

# changing format for gears model
if not 'ctrl' in adata.obs['condition'].values:
    print("Warning: 'ctrl' already present in conditions, renaming might cause issues.")
    # mask for 'non-targeting'
    mask_ctrl = adata.obs['condition'] == 'non-targeting'
    # changing values: 
    adata.obs.loc[mask_ctrl, 'condition'] = 'ctrl'
    adata.obs.loc[~mask_ctrl, 'condition'] = adata.obs.loc[~mask_ctrl, 'condition'] + '+ctrl'


# renaming columns
adata.obs['cell_type'] = 'H1'
adata.var.rename(columns={'gene_id': 'gene_name'}, inplace=True)

In [ ]:
adata

Different versions of models

1. Default 

2. Different correlations 

In [ ]:
# model 1 - default
model_name = Path('model_1-default')

In [ ]:
# model 2
model_name = Path('model_2-')
#TODO zmienic te wersje sprawdzić jak się globalnie zachowują 

In [ ]:
# paths
model_path = Path(gears_data_path, model_name) 

### Model training

### 3.10 Library adjustment 
in file:
```Bash
~/.pyenv/versions/scRNA_venv/lib/python3.10/site-packages/gears/gears.py
```
We need to change line 87 from:
```Python
np.mean(self.adata.X[(self.adata.obs.condition == 'ctrl')],
```
to:
```Python
np.mean(self.adata.X[(self.adata.obs.condition == 'ctrl').values],
```
Because we plugging data seriers into numpy array and they are not compatible 

### Idea

We create two dataloaders
1. ALl data -> training 
2. Half data for test -> for later metrics

In [ ]:
n=500

torch.tensor(
np.mean(adata.X[(adata.obs.condition == 'ctrl').to_numpy()], axis=0)).reshape(-1,)


# self.ctrl_expression = torch.tensor(
#             np.mean(self.adata.X[(self.adata.obs.condition == 'ctrl').to_numpy()],
#                     axis=0)).reshape(-1, ).to(self.device)

version from 10.3 (same on server) - nie działa model

```Python
 self.ctrl_expression = torch.tensor(
            np.mean(self.adata.X[(self.adata.obs.condition == 'ctrl').values], #TODO rewritten 
                    axis=0)).reshape(-1, ).to(self.device)
        pert_full_id2pert = dict(self.adata.obs[['condition_name', 'condition']].values)
        self.dict_filter = {pert_full_id2pert[i]: j for i, j in
                            self.adata.uns['non_zeros_gene_idx'].items() if
                            i in pert_full_id2pert}
        self.ctrl_adata = self.adata[self.adata.obs['condition'] == 'ctrl']
```

In [ ]:
serjes = pd.Series([1, 2])
serjes.values
numpaj = np.array([[1, 1, 0, 1], [1, 0, 1, 0]])
# serjes, numpaj[serjes==1]
(serjes==1).to_numpy()

In [ ]:
## creating and loading data in gears format. 

# parameters
dataset_name = "vcc_data"
want_process_data = False

# process
pert_data = PertData(loader_path)
## preparing data
if want_process_data:
    pert_data.new_data_process(dataset_name=dataset_name, adata=adata)
## loading data
pert_data.load(data_path=str(loader_path)+f'/{dataset_name}')
# creating masks for training and validation
pert_data.prepare_split(split='simulation', train_gene_set_size=0.95)


# creating data loader for training 
pert_data.get_dataloader(batch_size = 32, test_batch_size = 128)


In [ ]:
# training model
gears_model = GEARS(pert_data, device = 'cuda', 
                        weight_bias_track = False, 
                        proj_name = str(model_name), 
                        exp_name = str(dataset_name))
gears_model.model_initialize(hidden_size = 32)
# training model
gears_model.train(epochs = 20, lr = 1e-3)
# saving model & loading model
gears_model.save_model(model_path)

In [ ]:
# testing model - tutaj trzeba załadować model i na drugi data loaderze to zrobić. 
gears_model.load_pretrained('test_model')

In [ ]:
# testing for VCC

PAMIĘTAĆ !!!

- Ściągnąć wszystko na komputer, żeby nie musieć trenować jeszcze raz tylko to byśmy uplodowali (około 1.5h pobierania) - 5mb/s 
- nie musimy statystyk modelu sprawdzać, musimy przygotować data sety (najpierw  żeby to pobrać jeszcze) 
- i wytrenować modele 
